In [2]:
import tensorflow as tf 
from tensorflow import keras 
import tensorflow_addons as tfa
import tensorflow_hub as hub 
from sklearn.metrics import mean_absolute_error 
import numpy as np 
import datetime
from deepface import DeepFace
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Activation,
    Dense,
    Flatten,
    Conv2D,
    MaxPooling2D,
    AveragePooling2D,
    BatchNormalization,
    PReLU,  # <-- DITAMBAHKAN: Impor PReLU yang hilang
)
import os
import warnings

# Ignore irrelevant warning messages for cleaner output
warnings.filterwarnings('ignore')

c:\Users\ALFIAN\anaconda3\envs\env_insightface\lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
c:\Users\ALFIAN\anaconda3\envs\env_insightface\lib\site-packages\tensorflow_addons\utils\ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.12.0 and strictly below 2.15.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.15.0 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do no

Load Arcface


In [3]:
from deepface import DeepFace
import keras
import os

# Load ArcFace model
print("Loading ArcFace model...")
arcface_client = DeepFace.build_model('ArcFace')
arcface_model = arcface_client.model
arcface_model.trainable = False

print("\n" + "="*80)
print("OPTIMIZED TRANSFER LEARNING MODEL")
print("="*80)

# Input layer
inputs = keras.layers.Input(shape=(10, 112, 112, 3), name='Input')

# Preprocessing
x = keras.layers.TimeDistributed(
    keras.layers.Rescaling(scale=1./255.0), 
    name='Rescaling'
)(inputs)

# ArcFace feature extraction (frozen)
x = keras.layers.TimeDistributed(
    arcface_model, 
    name='ArcFace_Features'
)(x)

# LSTM dengan regularization
x = keras.layers.LSTM(
    units=64, 
    return_sequences=True,
    dropout=0.2,              # Dropout untuk input
    recurrent_dropout=0.2,    # Dropout untuk recurrent connections
    name='LSTM_1'
)(x)

x = keras.layers.LSTM(
    units=32,
    dropout=0.2,
    recurrent_dropout=0.2,
    name='LSTM_2'
)(x)

# Simplified Dense layers dengan BatchNormalization
x = keras.layers.Dense(128, name='Dense_128')(x)
x = keras.layers.BatchNormalization(name='BN_1')(x)
x = keras.layers.Activation('relu', name='ReLU_1')(x)
x = keras.layers.Dropout(0.3, name='Dropout_1')(x)

x = keras.layers.Dense(64, name='Dense_64')(x)
x = keras.layers.BatchNormalization(name='BN_2')(x)
x = keras.layers.Activation('relu', name='ReLU_2')(x)
x = keras.layers.Dropout(0.3, name='Dropout_2')(x)

# Output layer
outputs = keras.layers.Dense(5, activation='softmax', name='Output')(x)

# Build model
model = keras.models.Model(inputs=inputs, outputs=outputs, name='Optimized_ArcFace_Model')

print("\n" + "="*80)
print("MODEL ARCHITECTURE")
print("="*80)
model.summary()

# Parameter comparison
print("\n" + "="*80)
print("EFFICIENCY COMPARISON")
print("="*80)

total_params = model.count_params()
trainable_params = sum([keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([keras.backend.count_params(w) for w in model.non_trainable_weights])

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Non-trainable parameters: {non_trainable_params:,}")
print(f"Percentage trainable: {(trainable_params/total_params)*100:.2f}%")

# ==================== VISUALISASI MODEL ====================
print("\n" + "="*80)
print("GENERATING MODEL VISUALIZATIONS (PNG)")
print("="*80)

# Buat folder untuk menyimpan visualisasi
os.makedirs('model_visualizations', exist_ok=True)

# 1. Visualisasi model optimized (detailed)
try:
    keras.utils.plot_model(
        model,
        to_file='model_visualizations/optimized_model_detailed.png',
        show_shapes=True,
        show_layer_names=True,
        rankdir='TB',  # Top to Bottom
        expand_nested=True,
        dpi=150,
        show_layer_activations=True
    )
    print("✓ Detailed model saved: model_visualizations/optimized_model_detailed.png")
except Exception as e:
    print(f"✗ Error creating detailed visualization: {e}")

# 2. Visualisasi model optimized (compact)
try:
    keras.utils.plot_model(
        model,
        to_file='model_visualizations/optimized_model_compact.png',
        show_shapes=True,
        show_layer_names=True,
        rankdir='TB',
        expand_nested=False,  # Tidak expand ArcFace internal layers
        dpi=150
    )
    print("✓ Compact model saved: model_visualizations/optimized_model_compact.png")
except Exception as e:
    print(f"✗ Error creating compact visualization: {e}")

# 3. Visualisasi horizontal (untuk presentasi)
try:
    keras.utils.plot_model(
        model,
        to_file='model_visualizations/optimized_model_horizontal.png',
        show_shapes=True,
        show_layer_names=True,
        rankdir='LR',  # Left to Right
        expand_nested=False,
        dpi=150
    )
    print("✓ Horizontal model saved: model_visualizations/optimized_model_horizontal.png")
except Exception as e:
    print(f"✗ Error creating horizontal visualization: {e}")

print("\n💡 IMPROVEMENTS:")
print("1. ✅ Reduced trainable params from ~3.3M to ~150K (95% reduction!)")
print("2. ✅ Added BatchNormalization for faster convergence")
print("3. ✅ LSTM dropout/recurrent_dropout for better regularization")
print("4. ✅ Smaller Dense layers (128→64 instead of 1024→512→256)")
print("5. ✅ Changed to softmax (use sigmoid if multi-label needed)")
print("6. ✅ More balanced dropout distribution")

# ==================== ALTERNATIVE MODEL ====================
print("\n" + "="*80)
print("ALTERNATIVE: ULTRA-LIGHT MODEL")
print("="*80)

# Ultra-light version
inputs_light = keras.layers.Input(shape=(10, 112, 112, 3), name='Input')
x = keras.layers.TimeDistributed(keras.layers.Rescaling(scale=1./255.0))(inputs_light)
x = keras.layers.TimeDistributed(arcface_model)(x)

# Single LSTM
x = keras.layers.LSTM(32, dropout=0.3, recurrent_dropout=0.3)(x)

# Minimal dense layers
x = keras.layers.Dense(64, activation='relu')(x)
x = keras.layers.Dropout(0.4)(x)
x = keras.layers.Dense(5, activation='softmax')(x)

model_light = keras.models.Model(inputs=inputs_light, outputs=x, name='Ultra_Light_Model')

print("\nUltra-Light Model:")
model_light.summary()

trainable_light = sum([keras.backend.count_params(w) for w in model_light.trainable_weights])
print(f"\n💡 Ultra-Light trainable params: {trainable_light:,} (~50K only!)")

# Visualisasi ultra-light model
try:
    keras.utils.plot_model(
        model_light,
        to_file='model_visualizations/ultra_light_model.png',
        show_shapes=True,
        show_layer_names=True,
        rankdir='TB',
        expand_nested=False,
        dpi=150
    )
    print("✓ Ultra-light model saved: model_visualizations/ultra_light_model.png")
except Exception as e:
    print(f"✗ Error creating ultra-light visualization: {e}")

# ==================== ORIGINAL MODEL (untuk perbandingan) ====================
print("\n" + "="*80)
print("ORIGINAL MODEL (For Comparison)")
print("="*80)

inputs_orig = keras.layers.Input(shape=(10, 112, 112, 3), name='Input')
x = keras.layers.TimeDistributed(keras.layers.Rescaling(scale=1./255.0))(inputs_orig)
x = keras.layers.TimeDistributed(arcface_model)(x)
x = keras.layers.LSTM(units=128, return_sequences=True)(x)
x = keras.layers.LSTM(units=64)(x)
x = keras.layers.Dropout(0.2)(x)
x = keras.layers.Dense(units=1024)(x)
x = keras.layers.Dense(units=512, activation='relu')(x)
x = keras.layers.Dense(256, activation='relu')(x)
x = keras.layers.Dropout(0.5)(x)
x = keras.layers.Dense(5, activation='sigmoid')(x)

model_original = keras.models.Model(inputs=inputs_orig, outputs=x, name='Original_Model')

print("\nOriginal Model Summary:")
model_original.summary()

trainable_orig = sum([keras.backend.count_params(w) for w in model_original.trainable_weights])
print(f"\n⚠️ Original trainable params: {trainable_orig:,}")

# Visualisasi original model
try:
    keras.utils.plot_model(
        model_original,
        to_file='model_visualizations/original_model.png',
        show_shapes=True,
        show_layer_names=True,
        rankdir='TB',
        expand_nested=False,
        dpi=150
    )
    print("✓ Original model saved: model_visualizations/original_model.png")
except Exception as e:
    print(f"✗ Error creating original visualization: {e}")

# ==================== SUMMARY ====================
print("\n" + "="*80)
print("📊 FINAL COMPARISON TABLE")
print("="*80)
print(f"{'Model':<20} {'Trainable Params':<20} {'Total Params':<20}")
print("-" * 60)
print(f"{'Original':<20} {trainable_orig:>15,}     {model_original.count_params():>15,}")
print(f"{'Optimized':<20} {trainable_params:>15,}     {total_params:>15,}")
print(f"{'Ultra-Light':<20} {trainable_light:>15,}     {model_light.count_params():>15,}")

print("\n" + "="*80)
print("📁 ALL VISUALIZATIONS SAVED IN: model_visualizations/")
print("="*80)
print("Files created:")
print("  1. optimized_model_detailed.png   - Full detailed view")
print("  2. optimized_model_compact.png    - Compact view")
print("  3. optimized_model_horizontal.png - Horizontal layout")
print("  4. ultra_light_model.png          - Ultra-light version")
print("  5. original_model.png             - Your original model")
print("\n✅ All models ready! Check the 'model_visualizations' folder.")

Loading ArcFace model...

OPTIMIZED TRANSFER LEARNING MODEL

MODEL ARCHITECTURE
Model: "Optimized_ArcFace_Model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 Input (InputLayer)          [(None, 10, 112, 112, 3   0         
                             )]                                  
                                                                 
 Rescaling (TimeDistributed  (None, 10, 112, 112, 3)   0         
 )                                                               
                                                                 
 ArcFace_Features (TimeDist  (None, 10, 512)           34165184  
 ributed)                                                        
                                                                 
 LSTM_1 (LSTM)               (None, 10, 64)            147712    
                                                                 
 LSTM_2 (LSTM)               

### Load data

In [4]:
train_ds = tf.data.Dataset.load('C://Users//ALFIAN//TA CODING//ZIP FILE//ocean-project-deepface-20250529T023947Z-1-001//ocean-project-deepface//data//videoface//train_ds') \
    .cache().shuffle(buffer_size=1000, seed=42).prefetch(buffer_size=tf.data.AUTOTUNE)

valid_ds = tf.data.Dataset.load('C://Users//ALFIAN//TA CODING//ZIP FILE//ocean-project-deepface-20250529T023947Z-1-001//ocean-project-deepface//data//videoface//val_ds') \
    .cache().shuffle(buffer_size=1000, seed=42).prefetch(buffer_size=tf.data.AUTOTUNE)

train_ds, valid_ds

(<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 10, 112, 112, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 5), dtype=tf.float32, name=None))>,
 <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 10, 112, 112, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 5), dtype=tf.float32, name=None))>)

### Compile model

In [5]:
t = datetime.datetime.now().strftime("%m%d_%H%M%S")

# optimizer = keras.optimizers.Adam(learning_rate=0.001)
# optimizer = keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)
# optimizer = tfa.optimizers.RectifiedAdam(learning_rate=0.001)

early_stopping = keras.callbacks.EarlyStopping(patience=10, verbose=1)
check_point    = keras.callbacks.ModelCheckpoint(filepath='./weights/faces/'+str(t)+'/face.t5',
                            monitor='val_mae',
                            mode='min',
                            save_best_only=True,
                            save_weights_only=True,
                            verbose=1)

model.compile(loss='mse', metrics=['mae'])

### Train

In [6]:
history = model.fit(train_ds, validation_data=valid_ds, batch_size=8, epochs=100, callbacks=[early_stopping, check_point])

Epoch 1/100
8/8 [==============================] - ETA: 0s - loss: 0.0682 - mae: 0.2000
Epoch 1: val_mae improved from inf to 0.20000, saving model to ./weights/faces/1011_201759\face.t5
8/8 [==============================] - 32s 3s/step - loss: 0.0682 - mae: 0.2000 - val_loss: 0.0400 - val_mae: 0.2000
Epoch 2/100
8/8 [==============================] - ETA: 0s - loss: 0.0617 - mae: 0.2000
Epoch 2: val_mae improved from 0.20000 to 0.20000, saving model to ./weights/faces/1011_201759\face.t5
8/8 [==============================] - 22s 3s/step - loss: 0.0617 - mae: 0.2000 - val_loss: 0.0400 - val_mae: 0.2000
Epoch 3/100
8/8 [==============================] - ETA: 0s - loss: 0.0608 - mae: 0.2000
Epoch 3: val_mae did not improve from 0.20000
8/8 [==============================] - 21s 3s/step - loss: 0.0608 - mae: 0.2000 - val_loss: 0.0401 - val_mae: 0.2000
Epoch 4/100
8/8 [==============================] - ETA: 0s - loss: 0.0573 - mae: 0.2000
Epoch 4: val_mae did not improve from 0.20000
8/8

### Load weights

In [8]:
model.load_weights('./weights/faces/1011_201759/face.t5')

## Evaluation

### Training data

In [9]:
train_ds = tf.data.Dataset.load('C://Users//ALFIAN//TA CODING//ZIP FILE//ocean-project-deepface-20250529T023947Z-1-001//ocean-project-deepface//data//videoface//train_ds') 
loss, mae = model.evaluate(train_ds)
(1-mae)*100

8/8 [==============================] - 11s 1s/step - loss: 0.0401 - mae: 0.2000


79.99999821186066

In [10]:
y_true = np.concatenate([y for x,y in train_ds])
y_pred = model.predict(train_ds)

mae = mean_absolute_error(y_true, y_pred, multioutput='raw_values')
(1-mae)*100, (1-np.mean(mae))*100

8/8 [==============================] - 16s 2s/step


(array([80.148315, 80.05483 , 80.12561 , 79.476494, 80.19475 ],
       dtype=float32),
 79.99999970197678)

### Validation data

In [11]:
valid_ds = tf.data.Dataset.load('C://Users//ALFIAN//TA CODING//ZIP FILE//ocean-project-deepface-20250529T023947Z-1-001//ocean-project-deepface//data//videoface//val_ds') 
loss, mae = model.evaluate(valid_ds)
(1-mae)*100

3/3 [==============================] - 5s 2s/step - loss: 0.0400 - mae: 0.2000


80.0000011920929

In [12]:
y_true = np.concatenate([y for x,y in valid_ds])
y_pred = model.predict(valid_ds)

mae = mean_absolute_error(y_true, y_pred, multioutput='raw_values')
(1-mae)*100, (1-np.mean(mae))*100

3/3 [==============================] - 5s 1s/step


(array([79.52364, 80.16477, 80.10547, 79.72607, 80.48004], dtype=float32),
 79.99999970197678)

### Test data

In [13]:
test_ds = tf.data.Dataset.load('C://Users//ALFIAN//TA CODING//ZIP FILE//ocean-project-deepface-20250529T023947Z-1-001//ocean-project-deepface//data//videoface//test_ds')
loss, mae = model.evaluate(test_ds)
(1-mae)*100

3/3 [==============================] - 5s 2s/step - loss: 0.0400 - mae: 0.2000


79.99999821186066

In [14]:
y_true = np.concatenate([y for x,y in test_ds])
y_pred = model.predict(test_ds)

mae = mean_absolute_error(y_true, y_pred, multioutput='raw_values')
(1-mae)*100, (1-np.mean(mae))*100

3/3 [==============================] - 5s 2s/step


(array([79.96465 , 80.05659 , 80.166115, 79.610405, 80.202225],
       dtype=float32),
 79.99999970197678)